# 🌫️ Weather & Air Quality Data Integration
### Albany & Syracuse, New York (2023–2025)
**Author:** Aby Joe Jose  

---

## Project Overview

This project integrates **three public data sources** via REST APIs to build a unified, analysis-ready dataset linking daily meteorological conditions with PM2.5 air quality measurements across two New York cities.

| Source | API | Data |
|--------|-----|------|
| EPA AQS | `aqs.epa.gov` | Daily PM2.5 concentration & AQI |
| NOAA CDO | `ncdc.noaa.gov` | Temperature, precipitation, wind speed, humidity, snow depth |
| Open-Meteo Archive | `archive-api.open-meteo.com` | Dominant daily wind direction |

**Pipeline:**  
`API Extraction` → `Data Cleaning & Merging` → `MySQL Database` → `Analysis & Visualization`

---

## 1. Imports & Configuration

In [ ]:
import requests
import pandas as pd
import pymysql
import csv
import yaml
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

print("Libraries loaded successfully.")

## 2. Load Configuration

Credentials and API keys are stored in `config.yml` (excluded from version control via `.gitignore`).  
See `example_config.yml` for the required format.


In [ ]:
# Load database credentials and API keys from config.yml
with open('config.yml') as f:
    config = yaml.safe_load(f)

db     = config['db']
EPA_EMAIL   = config['epa']['email']
EPA_KEY     = config['epa']['key']
NOAA_TOKEN  = config['noaa']['token']

print("Config loaded.")
print(f"DB host  : {db['host']}")
print(f"EPA email: {EPA_EMAIL}")

---
## 3. Data Extraction

### 3.1 EPA Air Quality System (AQS) — PM2.5 Data

**API:** `https://aqs.epa.gov/data/api/dailyData/byCounty`  
**Parameter:** `88101` (PM2.5 — Local Conditions)  
**Cities:** Albany (County 001) and Syracuse (County 067), New York (State 36)  
**Period:** 2023–2025


In [ ]:
param = "88101"   # PM2.5 — Local Conditions

locations = {
    "Albany, NY"  : ("36", "001"),
    "Syracuse, NY": ("36", "067"),
}

date_ranges = [
    ("20230101", "20231231"),
    ("20240101", "20241231"),
    ("20250101", "20251201"),
]

all_epa_data = []

for city, (state, county) in locations.items():
    for startdate, enddate in date_ranges:
        params = {
            "email" : EPA_EMAIL,
            "key"   : EPA_KEY,
            "param" : param,
            "bdate" : startdate,
            "edate" : enddate,
            "state" : state,
            "county": county,
        }
        response = requests.get(
            "https://aqs.epa.gov/data/api/dailyData/byCounty",
            params=params
        )
        data = response.json()
        if "Data" in data:
            df_chunk = pd.DataFrame(data["Data"])
            df_chunk["city"] = city
            all_epa_data.append(df_chunk)
            print(f"  ✅ {city} | {startdate[:4]} | {len(df_chunk)} records")
        else:
            print(f"  ⚠️  No data: {city} | {startdate[:4]}")

print(f"\nTotal chunks collected: {len(all_epa_data)}")

#### Clean & Save PM2.5 Data

In [ ]:
pm25_df = pd.concat(all_epa_data, ignore_index=True)
pm25_df["date_local"] = pd.to_datetime(pm25_df["date_local"])

# Keep only relevant columns and rename
pm25_df = pm25_df[['date_local', 'arithmetic_mean', 'aqi', 'state', 'county']]
pm25_df.rename(columns={'arithmetic_mean': 'PM25'}, inplace=True)

# One record per city per day (take first reading)
pm25_df = pm25_df.groupby(['date_local', 'county'], as_index=False).first()

# Map county name to city
pm25_df["city"] = pm25_df["county"].replace({"Onondaga": "Syracuse"})
pm25_df = pm25_df.drop('county', axis=1)

pm25_df.to_csv("pm25_by_county_23_25.csv", index=False)

print(f"PM2.5 records: {len(pm25_df)}")
print(f"Date range   : {pm25_df['date_local'].min().date()} → {pm25_df['date_local'].max().date()}")
pm25_df.head()

---
### 3.2 NOAA Climate Data Online (CDO) — Weather Variables

**API:** `https://www.ncdc.noaa.gov/cdo-web/api/v2/data`  
**Dataset:** GHCND (Global Historical Climatology Network Daily)  

| Variable | Description |
|----------|-------------|
| TMAX | Maximum daily temperature |
| TMIN | Minimum daily temperature |
| PRCP | Precipitation |
| SNWD | Snow depth |
| AWND | Average wind speed |
| RHMN | Minimum relative humidity |


In [ ]:
stations_dict = {
    "Albany"  : "GHCND:USW00014735",
    "Syracuse": "GHCND:USW00014771",
}

datatypeid  = ["TMAX", "TMIN", "PRCP", "SNWD", "AWND", "RHMN"]
date_ranges = [
    ("2023-01-01", "2023-12-31"),
    ("2024-01-01", "2024-12-31"),
    ("2025-01-01", "2025-12-01"),
]

headers      = {"token": NOAA_TOKEN}
all_noaa     = []

for city, stationid in stations_dict.items():
    for startdate, enddate in date_ranges:
        limit  = 1000
        offset = 1

        while True:
            params = {
                "datasetid" : "GHCND",
                "stationid" : stationid,
                "startdate" : startdate,
                "enddate"   : enddate,
                "datatypeid": datatypeid,
                "limit"     : limit,
                "offset"    : offset,
                "units"     : "standard",
            }
            response = requests.get(
                "https://www.ncdc.noaa.gov/cdo-web/api/v2/data",
                headers=headers,
                params=params
            )
            data = response.json()
            results = data.get("results", [])
            for rec in results:
                rec["city"] = city
            all_noaa.extend(results)

            if len(results) < limit:
                break
            offset += limit

        print(f"  ✅ {city} | {startdate[:4]}")

print(f"\nTotal NOAA records: {len(all_noaa)}")

#### Pivot NOAA data — one row per city per day

In [ ]:
noaa_df = pd.DataFrame(all_noaa)
noaa_df["date"] = pd.to_datetime(noaa_df["date"])

# Pivot: date × city → AWND, PRCP, RHMN, SNWD, TMAX, TMIN columns
df_pivot = noaa_df.pivot_table(
    index=["date", "city"],
    columns="datatype",
    values="value"
)
df_pivot.reset_index(inplace=True)
df_pivot.columns.name = None

df_pivot.to_csv('weather_data_23-25.csv', index=False)

print(f"Weather records: {len(df_pivot)}")
print(f"Columns: {list(df_pivot.columns)}")
df_pivot.head()

---
### 3.3 Open-Meteo Archive API — Wind Direction

NOAA does not provide dominant daily wind direction in an accessible form.  
**API:** `https://archive-api.open-meteo.com/v1/archive` (free, no key required)  
**Variable:** `wind_direction_10m_dominant` — daily dominant wind direction in degrees (0°=N, 90°=E, 180°=S, 270°=W)


In [ ]:
cities = {
    "Syracuse": {"lat": 43.0481, "lon": -76.1474},
    "Albany"  : {"lat": 42.6526, "lon": -73.7562},
}

start_date = "2023-01-01"
end_date   = "2025-12-01"
all_wind   = []

for city, coords in cities.items():
    params = {
        "latitude"  : coords["lat"],
        "longitude" : coords["lon"],
        "start_date": start_date,
        "end_date"  : end_date,
        "daily"     : "wind_direction_10m_dominant",
        "timezone"  : "auto",
    }
    response = requests.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params=params
    )
    daily = response.json().get("daily", {})
    for d, wd in zip(daily.get("time", []), daily.get("wind_direction_10m_dominant", [])):
        all_wind.append({"date": d, "city": city, "wind_direction": wd})
    print(f"  ✅ {city} | {len(daily.get('time', []))} records")

wind_df = pd.DataFrame(all_wind).sort_values(["date", "city"]).reset_index(drop=True)
wind_df.to_csv('Wind_direction.csv', index=False)

print(f"\nWind direction records: {len(wind_df)}")
wind_df.head()

---
## 4. Data Processing & Merging

Merge all three sources on `date` and `city` to create a single unified dataset.


In [ ]:
# Load saved CSVs
weather_df = pd.read_csv('weather_data_23-25.csv', index_col=False)
pm25_df    = pd.read_csv('pm25_by_county_23_25.csv', index_col=False)
wind_df    = pd.read_csv('Wind_direction.csv', index_col=False)

# Standardise column names
pm25_df.rename(columns={'date_local': 'date', 'county': 'city'}, inplace=True)

# Parse dates
for df_ in [weather_df, pm25_df, wind_df]:
    df_["date"] = pd.to_datetime(df_["date"])

# Merge: weather + PM2.5 → add wind direction
df_merged = pd.merge(weather_df, pm25_df, on=['date', 'city'], how='inner')
df_merged = pd.merge(df_merged, wind_df,  on=['date', 'city'], how='inner')

df_merged.to_csv('All_data.csv', index=False)

print(f"Merged dataset shape: {df_merged.shape}")
print(f"Date range          : {df_merged['date'].min().date()} → {df_merged['date'].max().date()}")
print(f"Cities              : {df_merged['city'].unique()}")
print(f"Missing values:\n{df_merged.isnull().sum()}")
df_merged.head()

---
## 5. MySQL Database — Storage

### Relational Schema

The table uses a **composite unique key** on `(date, city)` to prevent duplicate records and supports incremental inserts.

```
weather_air_quality
─────────────────────────────────────────
tid               INT  PK AUTO_INCREMENT
date              DATE
city              VARCHAR(25)
state             VARCHAR(50)
wind_speed        FLOAT        (AWND)
precipitation     FLOAT        (PRCP)
relative_humidity FLOAT        (RHMN)
snow              FLOAT        (SNWD)
temperature_max   FLOAT        (TMAX)
temperature_min   FLOAT        (TMIN)
pm25              FLOAT
aqi               FLOAT
wind_direction    INT
UNIQUE KEY (date, city)
─────────────────────────────────────────
```


In [ ]:
# Connect to MySQL using credentials from config.yml
conn = pymysql.connect(
    host     = db['host'],
    port     = 3306,
    user     = db['user'],
    passwd   = db['pw'],
    db       = db['db'],
    autocommit = True
)
cur = conn.cursor(pymysql.cursors.DictCursor)
print("Connected to MySQL:", db['host'])

In [ ]:
# Create table (drop if exists for clean reload)
cur.execute('DROP TABLE IF EXISTS `weather_air_quality`;')

create_table_sql = """
CREATE TABLE `weather_air_quality` (
    `tid`               INT AUTO_INCREMENT PRIMARY KEY,
    `date`              DATE NOT NULL,
    `city`              VARCHAR(25) NOT NULL,
    `state`             VARCHAR(50),
    `wind_speed`        FLOAT,
    `precipitation`     FLOAT,
    `relative_humidity` FLOAT,
    `snow`              FLOAT,
    `temperature_max`   FLOAT,
    `temperature_min`   FLOAT,
    `pm25`              FLOAT,
    `aqi`               FLOAT,
    `wind_direction`    INT,
    UNIQUE KEY uniq_weather (`date`, `city`)
);
"""
cur.execute(create_table_sql)
print("Table created: weather_air_quality")

In [ ]:
def clean_float(val):
    """Safely convert a value to float; return None for missing/invalid."""
    try:
        if val in ('', 'NA', 'N/A', 'null', None):
            return None
        return float(val)
    except:
        return None

# Build insert data from merged CSV
insert_data = []
with open('All_data.csv', 'r') as f:
    for row in csv.DictReader(f, skipinitialspace=True):
        insert_data.append((
            row['date'], row['city'], row.get('state'),
            clean_float(row['AWND']),  clean_float(row['PRCP']),
            clean_float(row['RHMN']),  clean_float(row['SNWD']),
            clean_float(row['TMAX']),  clean_float(row['TMIN']),
            clean_float(row['PM25']),  clean_float(row['aqi']),
            clean_float(row['wind_direction'])
        ))

insert_sql = """
INSERT INTO `weather_air_quality`(
    `date`, `city`, `state`,
    `wind_speed`, `precipitation`, `relative_humidity`, `snow`,
    `temperature_max`, `temperature_min`,
    `pm25`, `aqi`, `wind_direction`
) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE
    wind_speed        = VALUES(wind_speed),
    precipitation     = VALUES(precipitation),
    relative_humidity = VALUES(relative_humidity),
    pm25              = VALUES(pm25),
    aqi               = VALUES(aqi),
    wind_direction    = VALUES(wind_direction)
"""

cur.executemany(insert_sql, insert_data)
print(f"Inserted / updated {len(insert_data)} records into weather_air_quality.")

In [ ]:
# Verify record count in DB
cur.execute("SELECT city, COUNT(*) as records, MIN(date) as from_date, MAX(date) as to_date FROM weather_air_quality GROUP BY city;")
result = cur.fetchall()
print("\nDatabase summary:")
for row in result:
    print(f"  {row['city']:<12} | {row['records']} records | {row['from_date']} → {row['to_date']}")

---
## 6. Analysis & Visualization

### 6.1 Polar Plot — Wind Speed × Wind Direction × PM2.5

This plot encodes three variables simultaneously:
- **Angle (θ):** Wind direction in degrees (0°=N, 90°=E, 180°=S, 270°=W)
- **Radius (r):** Wind speed (AWND)
- **Color:** PM2.5 concentration (lighter = lower, darker = higher)


In [ ]:
df_plot = pd.read_csv('All_data.csv')
df_plot = df_plot.dropna(subset=['AWND', 'wind_direction', 'PM25'])

# Convert wind direction degrees to radians for polar plot
theta = np.deg2rad(df_plot['wind_direction'])
r     = df_plot['AWND']
pm25  = df_plot['PM25']

fig = plt.figure(figsize=(9, 9))
ax  = fig.add_subplot(111, polar=True)

sc = ax.scatter(
    theta, r,
    c=pm25, cmap='YlOrRd',
    alpha=0.6, s=15, edgecolors='none'
)

ax.set_theta_zero_location('N')   # 0° at top (North)
ax.set_theta_direction(-1)         # Clockwise
ax.set_xlabel('Wind Speed (AWND)', labelpad=15)
ax.set_title('Wind Speed × Wind Direction × PM2.5\n(Albany & Syracuse, NY  2023–2025)',
             pad=20, fontsize=13, fontweight='bold')

cbar = plt.colorbar(sc, ax=ax, pad=0.1, fraction=0.04)
cbar.set_label('PM2.5 (µg/m³)', fontsize=11)

plt.tight_layout()
plt.savefig('images/Polar_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Key finding: High PM2.5 clusters near the centre (low wind speed) — stagnant air traps particulates.")

### 6.2 PM2.5 vs Relative Humidity

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for city, color in zip(['Albany', 'Syracuse'], ['steelblue', 'darkorange']):
    subset = df_plot[df_plot['city'] == city].dropna(subset=['RHMN', 'PM25'])
    ax.scatter(subset['RHMN'], subset['PM25'],
               alpha=0.4, s=12, color=color, label=city, edgecolors='none')

ax.set_xlabel('Relative Humidity (%)')
ax.set_ylabel('PM2.5 (µg/m³)')
ax.set_title('PM2.5 vs Relative Humidity', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('images/Plot1.png', dpi=150, bbox_inches='tight')
plt.show()
print("Finding: No meaningful correlation between relative humidity and PM2.5.")

### 6.3 AQI vs Relative Humidity

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for city, color in zip(['Albany', 'Syracuse'], ['steelblue', 'darkorange']):
    subset = df_plot[df_plot['city'] == city].dropna(subset=['RHMN', 'aqi'])
    ax.scatter(subset['RHMN'], subset['aqi'],
               alpha=0.4, s=12, color=color, label=city, edgecolors='none')

ax.set_xlabel('Relative Humidity (%)')
ax.set_ylabel('AQI')
ax.set_title('AQI vs Relative Humidity', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('images/Plot2.png', dpi=150, bbox_inches='tight')
plt.show()
print("Finding: No meaningful correlation between relative humidity and AQI.")

### 6.4 PM2.5 Time Series — Albany vs Syracuse

In [ ]:
df_plot['date'] = pd.to_datetime(df_plot['date'])
df_ts = df_plot.groupby(['date', 'city'])['PM25'].mean().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))

for city, color in zip(['Albany', 'Syracuse'], ['steelblue', 'darkorange']):
    subset = df_ts[df_ts['city'] == city]
    ax.plot(subset['date'], subset['PM25'], alpha=0.7, linewidth=0.8,
            color=color, label=city)

ax.set_xlabel('Date')
ax.set_ylabel('PM2.5 (µg/m³)')
ax.set_title('Daily PM2.5 — Albany vs Syracuse (2023–2025)', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.5 Wind Speed vs PM2.5 — Seasonal Breakdown

In [ ]:
df_plot['month'] = pd.to_datetime(df_plot['date']).dt.month
df_plot['season'] = df_plot['month'].map({
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring',  4: 'Spring', 5: 'Spring',
    6: 'Summer',  7: 'Summer', 8: 'Summer',
    9: 'Fall',   10: 'Fall',  11: 'Fall'
})

fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharex=True, sharey=True)
seasons   = ['Winter', 'Spring', 'Summer', 'Fall']
colors    = ['#4e79a7', '#59a14f', '#f28e2b', '#e15759']

for ax, season, color in zip(axes.flatten(), seasons, colors):
    sub = df_plot[df_plot['season'] == season].dropna(subset=['AWND', 'PM25'])
    ax.scatter(sub['AWND'], sub['PM25'], alpha=0.4, s=12, color=color, edgecolors='none')
    ax.set_title(season, fontsize=12, fontweight='bold')
    ax.set_xlabel('Wind Speed (AWND)')
    ax.set_ylabel('PM2.5 (µg/m³)')
    ax.grid(True, alpha=0.3)

plt.suptitle('Wind Speed vs PM2.5 by Season', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("Finding: Low wind speed → elevated PM2.5 is consistent across all seasons.")

---
## 7. Summary & Key Findings

| Finding | Detail |
|---------|--------|
| **Wind speed is the strongest PM2.5 driver** | High PM2.5 consistently occurs at low wind speeds — stagnant air prevents dispersion |
| **Wind direction has minor influence** | North-easterly directions show slightly elevated PM2.5 |
| **Humidity has no correlation with PM2.5 or AQI** | Scatter plots show no discernible pattern |
| **Seasonal consistency** | The wind speed–PM2.5 relationship holds across all four seasons |
| **Data integration successful** | 3 APIs merged cleanly into a single normalized MySQL table |

---

**Tools:** Python · Pandas · Requests · PyMySQL · SQLAlchemy · Matplotlib · YAML  
**APIs:** EPA AQS · NOAA CDO · Open-Meteo Archive  
**Database:** MySQL (normalized schema, composite unique key, upsert logic)
